# Risk Management with FiveTwenty

This notebook demonstrates comprehensive risk management techniques for forex trading using the FiveTwenty.

## Prerequisites

1. Install dependencies: `pip install fivetwenty pandas numpy`
2. Set your OANDA API token: `FIVETWENTY_OANDA_TOKEN=your-token`
3. Use practice environment for learning

## Setup and Imports

In [ ]:
import os
from datetime import datetime

from fivetwenty import AsyncClient, Environment
from fivetwenty.exceptions import VeeTwentyError
from fivetwenty.models import TimeInForce

# Jupyter async support
try:
    import nest_asyncio

    nest_asyncio.apply()
except ImportError:
    print("Install nest_asyncio for Jupyter: pip install nest_asyncio")

# Configuration
TOKEN = os.getenv("FIVETWENTY_OANDA_TOKEN", "your-token-here")
ENVIRONMENT = Environment.PRACTICE
ACCOUNT_ID = None

print("✅ Setup complete" if TOKEN != "your-token-here" else "⚠️ Set FIVETWENTY_OANDA_TOKEN environment variable")

## Risk Management Framework

Let's create a comprehensive risk management class:

In [ ]:
class RiskManager:
    """Comprehensive risk management framework."""

    def __init__(self, client: AsyncClient, account_id: str):
        self.client = client
        self.account_id = account_id

        # Risk parameters (customize these based on your risk tolerance)
        self.max_risk_per_trade = 0.02  # 2% of account per trade
        self.max_daily_loss = 0.05  # 5% maximum daily loss
        self.max_portfolio_risk = 0.10  # 10% maximum portfolio risk
        self.max_drawdown = 0.15  # 15% maximum drawdown

        # Position limits
        self.max_positions = 5  # Maximum number of open positions
        self.max_correlation = 0.7  # Maximum correlation between positions

    async def get_account_info(self) -> dict:
        """Get current account information."""
        try:
            account = await self.client.accounts.get(self.account_id)
            return {
                "balance": float(account.balance),
                "nav": float(account.nav),
                "unrealized_pl": float(account.unrealized_pl),
                "margin_used": float(account.margin_used),
                "margin_available": float(account.margin_available),
                "open_trade_count": int(account.open_trade_count),
                "open_position_count": int(account.open_position_count),
                "currency": account.currency,
            }
        except VeeTwentyError as e:
            print(f"Error getting account info: {e.message}")
            return {}

    async def calculate_position_size(self, instrument: str, entry_price: float, stop_loss: float, risk_amount: float | None = None) -> int:
        """Calculate optimal position size based on risk management rules."""

        account_info = await self.get_account_info()
        if not account_info:
            return 0

        account_balance = account_info["balance"]

        # Use specified risk amount or default to max risk per trade
        if risk_amount is None:
            risk_amount = account_balance * self.max_risk_per_trade

        # Calculate pip value and risk in pips
        pip_difference = abs(entry_price - stop_loss)

        if pip_difference == 0:
            print("⚠️ Warning: Entry price equals stop loss price")
            return 0

        # Get pip value for the instrument
        pip_value = await self.get_pip_value(instrument, account_info["currency"])

        # Calculate position size
        position_size = int(risk_amount / (pip_difference * pip_value))

        print("📊 Position Size Calculation:")
        print(f"  Risk Amount: {risk_amount:.2f} {account_info['currency']}")
        print(f"  Pip Difference: {pip_difference:.5f}")
        print(f"  Pip Value: {pip_value:.2f}")
        print(f"  Calculated Size: {position_size} units")

        return position_size

    async def get_pip_value(self, instrument: str, account_currency: str) -> float:
        """Get pip value for position sizing calculations."""
        # Simplified pip value calculation
        # In practice, you'd want more sophisticated pip value calculation
        # based on current exchange rates

        if instrument.endswith("JPY"):
            return 0.01  # JPY pairs have different pip value
        return 0.0001  # Standard pip value for most pairs

    async def check_daily_loss_limit(self) -> bool:
        """Check if daily loss limit has been exceeded."""
        # Get today's transactions
        today = datetime.now().strftime("%Y-%m-%d")

        try:
            transactions = await self.client.transactions.list(account_id=self.account_id, from_time=f"{today}T00:00:00Z")

            # Calculate today's P/L
            daily_pl = 0.0
            for transaction in transactions.transactions:
                if hasattr(transaction, "pl") and transaction.pl:
                    daily_pl += float(transaction.pl)

            account_info = await self.get_account_info()
            account_balance = account_info["balance"]

            max_daily_loss_amount = account_balance * self.max_daily_loss

            print("📊 Daily Loss Check:")
            print(f"  Today's P/L: {daily_pl:.2f}")
            print(f"  Max Daily Loss: {max_daily_loss_amount:.2f}")

            if abs(daily_pl) > max_daily_loss_amount:
                print("🚨 Daily loss limit exceeded!")
                return False

            return True

        except VeeTwentyError as e:
            print(f"Error checking daily loss: {e.message}")
            return True  # Allow trading if we can't check

    async def check_portfolio_risk(self) -> dict:
        """Assess overall portfolio risk."""
        try:
            positions = await self.client.positions.list_open(self.account_id)
            account_info = await self.get_account_info()

            total_risk = 0.0
            position_details = []

            for position in positions:
                # Calculate risk for each position
                if position.long.units != "0":
                    unrealized_pl = float(position.long.unrealized_pl)
                    units = int(position.long.units)
                elif position.short.units != "0":
                    unrealized_pl = float(position.short.unrealized_pl)
                    units = int(position.short.units)
                else:
                    continue

                position_risk = abs(unrealized_pl) / account_info["balance"]
                total_risk += position_risk

                position_details.append({"instrument": position.instrument, "units": units, "unrealized_pl": unrealized_pl, "risk_percentage": position_risk * 100})

            risk_assessment = {"total_risk_percentage": total_risk * 100, "max_portfolio_risk_percentage": self.max_portfolio_risk * 100, "within_limits": total_risk <= self.max_portfolio_risk, "position_count": len(positions), "max_positions": self.max_positions, "position_details": position_details}

            return risk_assessment

        except VeeTwentyError as e:
            print(f"Error assessing portfolio risk: {e.message}")
            return {}

    async def validate_trade(self, instrument: str, units: int, entry_price: float, stop_loss: float, take_profit: float | None = None) -> dict:
        """Comprehensive trade validation before execution."""

        validation_result = {"approved": True, "warnings": [], "errors": [], "adjustments": {}}

        # Check daily loss limit
        if not await self.check_daily_loss_limit():
            validation_result["approved"] = False
            validation_result["errors"].append("Daily loss limit exceeded")

        # Check portfolio risk
        portfolio_risk = await self.check_portfolio_risk()
        if portfolio_risk and not portfolio_risk["within_limits"]:
            validation_result["warnings"].append(f"Portfolio risk ({portfolio_risk['total_risk_percentage']:.1f}%) exceeds limit")

        # Check position count
        if portfolio_risk and portfolio_risk["position_count"] >= self.max_positions:
            validation_result["approved"] = False
            validation_result["errors"].append(f"Maximum positions ({self.max_positions}) already reached")

        # Validate position size
        optimal_size = await self.calculate_position_size(instrument, entry_price, stop_loss)
        if abs(units) > optimal_size * 1.5:  # Allow 50% variance
            validation_result["warnings"].append(f"Position size ({abs(units)}) exceeds optimal size ({optimal_size})")
            validation_result["adjustments"]["suggested_units"] = optimal_size if units > 0 else -optimal_size

        # Risk-reward ratio check
        if take_profit:
            risk = abs(entry_price - stop_loss)
            reward = abs(take_profit - entry_price)
            risk_reward_ratio = reward / risk if risk > 0 else 0

            if risk_reward_ratio < 1.5:  # Minimum 1.5:1 risk-reward
                validation_result["warnings"].append(f"Risk-reward ratio ({risk_reward_ratio:.2f}) below recommended 1.5:1")

        return validation_result


print("✅ Risk management framework defined")

## Advanced Order Types for Risk Management

Let's create functions for advanced order types that help manage risk:

In [ ]:
class AdvancedOrders:
    """Advanced order management for risk control."""

    def __init__(self, client: AsyncClient, account_id: str):
        self.client = client
        self.account_id = account_id

    async def place_bracket_order(self, instrument: str, units: int, stop_loss_pips: float, take_profit_pips: float) -> dict:
        """Place a market order with automatic stop loss and take profit."""

        try:
            # Get current price
            prices = await self.client.pricing.get(account_id=self.account_id, instruments=[instrument])

            if not prices or not prices[0].asks or not prices[0].bids:
                return {"success": False, "error": "Could not get current price"}

            # Determine entry price based on direction
            if units > 0:  # Buy order
                entry_price = float(prices[0].asks[0].price)
                stop_loss_price = entry_price - (stop_loss_pips * 0.0001)
                take_profit_price = entry_price + (take_profit_pips * 0.0001)
            else:  # Sell order
                entry_price = float(prices[0].bids[0].price)
                stop_loss_price = entry_price + (stop_loss_pips * 0.0001)
                take_profit_price = entry_price - (take_profit_pips * 0.0001)

            # Adjust for JPY pairs
            if instrument.endswith("JPY"):
                stop_loss_price = entry_price + (stop_loss_pips * 0.01 * (-1 if units > 0 else 1))
                take_profit_price = entry_price + (take_profit_pips * 0.01 * (1 if units > 0 else -1))

            # Place bracket order
            response = await self.client.orders.create_market(account_id=self.account_id, instrument=instrument, units=units, stop_loss_on_fill={"price": f"{stop_loss_price:.5f}"}, take_profit_on_fill={"price": f"{take_profit_price:.5f}"})

            if response.order_fill_transaction:
                fill = response.order_fill_transaction
                return {"success": True, "trade_id": fill.trade_opened.trade_id if fill.trade_opened else None, "fill_price": float(fill.price), "stop_loss": stop_loss_price, "take_profit": take_profit_price, "units": int(fill.units)}
            return {"success": False, "error": "Order not filled"}

        except VeeTwentyError as e:
            return {"success": False, "error": f"OANDA error: {e.message}"}

    async def trailing_stop_loss(self, trade_id: str, trail_distance_pips: float) -> bool:
        """Set up a trailing stop loss for an existing trade."""

        try:
            # Get trade details
            trade = await self.client.trades.get(self.account_id, trade_id)

            if not trade:
                print(f"❌ Trade {trade_id} not found")
                return False

            # Calculate trailing distance
            trail_distance = trail_distance_pips * 0.01 if trade.instrument.endswith("JPY") else trail_distance_pips * 0.0001

            # Update stop loss with trailing distance
            await self.client.trades.update(account_id=self.account_id, trade_id=trade_id, stop_loss={"distance": f"{trail_distance:.5f}"})

            print(f"✅ Trailing stop loss set: {trail_distance_pips} pips")
            return True

        except VeeTwentyError as e:
            print(f"❌ Error setting trailing stop: {e.message}")
            return False

    async def scale_out_position(self, trade_id: str, scale_percentages: list[float], price_targets: list[float]) -> list[dict]:
        """Scale out of a position at multiple price levels."""

        if len(scale_percentages) != len(price_targets):
            print("❌ Scale percentages and price targets must have same length")
            return []

        try:
            # Get trade details
            trade = await self.client.trades.get(self.account_id, trade_id)

            if not trade:
                print(f"❌ Trade {trade_id} not found")
                return []

            original_units = int(trade.current_units)
            scale_orders = []

            for i, (percentage, target_price) in enumerate(zip(scale_percentages, price_targets, strict=False)):
                # Calculate units to close
                units_to_close = int(abs(original_units) * percentage / 100)
                if original_units < 0:
                    units_to_close = -units_to_close

                # Create limit order to close partial position
                order_response = await self.client.orders.create_limit(
                    account_id=self.account_id,
                    instrument=trade.instrument,
                    units=-units_to_close,  # Opposite direction to close
                    price=f"{target_price:.5f}",
                    time_in_force=TimeInForce.GTC,
                )

                if order_response.order_create_transaction:
                    order = order_response.order_create_transaction
                    scale_orders.append({"order_id": order.id, "percentage": percentage, "target_price": target_price, "units": units_to_close})
                    print(f"✅ Scale order {i + 1}: {percentage}% at {target_price}")

            return scale_orders

        except VeeTwentyError as e:
            print(f"❌ Error creating scale orders: {e.message}")
            return []


print("✅ Advanced orders class defined")

## Initialize Connection and Test Risk Management

In [ ]:
async def initialize_connection():
    """Initialize connection and get account ID."""
    global ACCOUNT_ID

    async with AsyncClient(token=TOKEN, environment=ENVIRONMENT) as client:
        try:
            accounts = await client.accounts.list()
            if accounts:
                ACCOUNT_ID = accounts[0].id
                print(f"✅ Connected to account: {ACCOUNT_ID}")
                return ACCOUNT_ID
            print("❌ No accounts found")
            return None
        except VeeTwentyError as e:
            print(f"❌ Connection error: {e.message}")
            return None


# Initialize connection
account_id = await initialize_connection()

## Portfolio Risk Assessment

In [ ]:
if account_id:
    async with AsyncClient(token=TOKEN, environment=ENVIRONMENT) as client:
        # Initialize risk manager
        risk_manager = RiskManager(client, account_id)

        print("📊 Current Account Status:")
        account_info = await risk_manager.get_account_info()

        if account_info:
            print(f"  Balance: {account_info['balance']:.2f} {account_info['currency']}")
            print(f"  NAV: {account_info['nav']:.2f} {account_info['currency']}")
            print(f"  Unrealized P/L: {account_info['unrealized_pl']:.2f} {account_info['currency']}")
            print(f"  Margin Used: {account_info['margin_used']:.2f} {account_info['currency']}")
            print(f"  Open Positions: {account_info['open_position_count']}")
            print(f"  Open Trades: {account_info['open_trade_count']}")

        print("\n🔍 Portfolio Risk Assessment:")
        portfolio_risk = await risk_manager.check_portfolio_risk()

        if portfolio_risk:
            print(f"  Total Portfolio Risk: {portfolio_risk['total_risk_percentage']:.2f}%")
            print(f"  Risk Limit: {portfolio_risk['max_portfolio_risk_percentage']:.2f}%")
            print(f"  Within Limits: {'✅' if portfolio_risk['within_limits'] else '❌'}")

            if portfolio_risk["position_details"]:
                print("\n  Position Details:")
                for pos in portfolio_risk["position_details"]:
                    print(f"    {pos['instrument']}: {pos['units']} units, P/L: {pos['unrealized_pl']:.2f}, Risk: {pos['risk_percentage']:.2f}%")
else:
    print("❌ No account connection - cannot assess risk")

## Position Sizing Calculation

In [ ]:
if account_id:
    async with AsyncClient(token=TOKEN, environment=ENVIRONMENT) as client:
        risk_manager = RiskManager(client, account_id)

        # Example position sizing calculation
        instrument = "EUR_USD"
        entry_price = 1.1000
        stop_loss = 1.0950  # 50 pip stop loss

        print(f"🧮 Position Sizing for {instrument}:")
        print(f"  Entry Price: {entry_price}")
        print(f"  Stop Loss: {stop_loss}")
        print(f"  Risk per Trade: {risk_manager.max_risk_per_trade * 100}%")

        optimal_size = await risk_manager.calculate_position_size(instrument, entry_price, stop_loss)

        print(f"\n✅ Optimal position size: {optimal_size} units")
else:
    print("❌ No account connection - cannot calculate position size")

## Trade Validation Example

In [ ]:
if account_id:
    async with AsyncClient(token=TOKEN, environment=ENVIRONMENT) as client:
        risk_manager = RiskManager(client, account_id)

        # Example trade validation
        trade_params = {"instrument": "GBP_USD", "units": 5000, "entry_price": 1.2500, "stop_loss": 1.2450, "take_profit": 1.2600}

        print("🔍 Validating Trade:")
        print(f"  Instrument: {trade_params['instrument']}")
        print(f"  Units: {trade_params['units']}")
        print(f"  Entry: {trade_params['entry_price']}")
        print(f"  Stop Loss: {trade_params['stop_loss']}")
        print(f"  Take Profit: {trade_params['take_profit']}")

        validation = await risk_manager.validate_trade(**trade_params)

        print("\n📋 Validation Result:")
        print(f"  Approved: {'✅' if validation['approved'] else '❌'}")

        if validation["errors"]:
            print("  🚨 Errors:")
            for error in validation["errors"]:
                print(f"    - {error}")

        if validation["warnings"]:
            print("  ⚠️ Warnings:")
            for warning in validation["warnings"]:
                print(f"    - {warning}")

        if validation["adjustments"]:
            print("  💡 Suggested Adjustments:")
            for key, value in validation["adjustments"].items():
                print(f"    - {key}: {value}")
else:
    print("❌ No account connection - cannot validate trade")

## Bracket Order Example

In [ ]:
if account_id:
    async with AsyncClient(token=TOKEN, environment=ENVIRONMENT) as client:
        advanced_orders = AdvancedOrders(client, account_id)

        print("🎯 Bracket Order Example (Demo - Uncomment to execute):")
        print("  Instrument: EUR_USD")
        print("  Units: 1000")
        print("  Stop Loss: 20 pips")
        print("  Take Profit: 40 pips")

        # Uncomment the lines below to place a real bracket order
        # bracket_result = await advanced_orders.place_bracket_order(
        #     instrument="EUR_USD",
        #     units=1000,
        #     stop_loss_pips=20,
        #     take_profit_pips=40
        # )
        #
        # if bracket_result['success']:
        #     print(f"✅ Bracket order placed successfully!")
        #     print(f"  Trade ID: {bracket_result['trade_id']}")
        #     print(f"  Fill Price: {bracket_result['fill_price']}")
        #     print(f"  Stop Loss: {bracket_result['stop_loss']}")
        #     print(f"  Take Profit: {bracket_result['take_profit']}")
        # else:
        #     print(f"❌ Bracket order failed: {bracket_result['error']}")

        print("\n💡 Uncomment the code above to place a real bracket order")
else:
    print("❌ No account connection - cannot place bracket order")

## Risk Monitoring Dashboard

In [ ]:
async def risk_dashboard(account_id: str):
    """Create a comprehensive risk monitoring dashboard."""

    if not account_id:
        print("❌ No account connection")
        return

    async with AsyncClient(token=TOKEN, environment=ENVIRONMENT) as client:
        risk_manager = RiskManager(client, account_id)

        print("🏛️ RISK MANAGEMENT DASHBOARD")
        print("=" * 50)

        # Account overview
        account_info = await risk_manager.get_account_info()
        if account_info:
            print("\n💰 Account Overview:")
            print(f"  Balance: {account_info['balance']:,.2f} {account_info['currency']}")
            print(f"  Equity (NAV): {account_info['nav']:,.2f} {account_info['currency']}")
            print(f"  Unrealized P/L: {account_info['unrealized_pl']:,.2f} {account_info['currency']}")

            # Calculate key ratios
            equity_ratio = (account_info["nav"] / account_info["balance"]) * 100
            margin_ratio = (account_info["margin_used"] / account_info["nav"]) * 100 if account_info["nav"] > 0 else 0

            print(f"  Equity Ratio: {equity_ratio:.2f}%")
            print(f"  Margin Usage: {margin_ratio:.2f}%")

        # Risk limits check
        print("\n🛡️ Risk Limits:")
        print(f"  Max Risk per Trade: {risk_manager.max_risk_per_trade * 100:.1f}%")
        print(f"  Max Daily Loss: {risk_manager.max_daily_loss * 100:.1f}%")
        print(f"  Max Portfolio Risk: {risk_manager.max_portfolio_risk * 100:.1f}%")
        print(f"  Max Positions: {risk_manager.max_positions}")

        # Daily loss check
        daily_ok = await risk_manager.check_daily_loss_limit()
        print(f"  Daily Loss Status: {'✅ OK' if daily_ok else '🚨 EXCEEDED'}")

        # Portfolio risk assessment
        portfolio_risk = await risk_manager.check_portfolio_risk()
        if portfolio_risk:
            print("\n📊 Portfolio Risk:")
            print(f"  Current Risk: {portfolio_risk['total_risk_percentage']:.2f}%")
            print(f"  Risk Status: {'✅ OK' if portfolio_risk['within_limits'] else '⚠️ HIGH'}")
            print(f"  Open Positions: {portfolio_risk['position_count']}/{portfolio_risk['max_positions']}")

            if portfolio_risk["position_details"]:
                print("\n📈 Active Positions:")
                for i, pos in enumerate(portfolio_risk["position_details"], 1):
                    status = "🟢" if pos["unrealized_pl"] >= 0 else "🔴"
                    print(f"  {i}. {pos['instrument']}: {pos['units']} units, P/L: {pos['unrealized_pl']:+.2f}, Risk: {pos['risk_percentage']:.1f}% {status}")

        print("\n📋 Risk Assessment Summary:")
        risk_score = 0

        if daily_ok:
            risk_score += 25
            print("  ✅ Daily loss within limits (+25 points)")
        else:
            print("  🚨 Daily loss exceeded (0 points)")

        if portfolio_risk and portfolio_risk["within_limits"]:
            risk_score += 25
            print("  ✅ Portfolio risk acceptable (+25 points)")
        else:
            print("  ⚠️ Portfolio risk elevated (+10 points)")
            risk_score += 10

        if account_info and equity_ratio >= 95:
            risk_score += 25
            print("  ✅ Strong equity position (+25 points)")
        elif account_info and equity_ratio >= 90:
            risk_score += 15
            print("  ⚠️ Moderate equity position (+15 points)")
        else:
            print("  🚨 Weak equity position (0 points)")

        if account_info and margin_ratio <= 50:
            risk_score += 25
            print("  ✅ Conservative margin usage (+25 points)")
        elif account_info and margin_ratio <= 75:
            risk_score += 15
            print("  ⚠️ Moderate margin usage (+15 points)")
        else:
            print("  🚨 High margin usage (0 points)")

        print(f"\n🎯 Overall Risk Score: {risk_score}/100")

        if risk_score >= 90:
            print("  🟢 EXCELLENT - Low risk, safe to trade")
        elif risk_score >= 70:
            print("  🟡 GOOD - Moderate risk, trade with caution")
        elif risk_score >= 50:
            print("  🟠 FAIR - Elevated risk, reduce position sizes")
        else:
            print("  🔴 POOR - High risk, consider closing positions")


# Run the risk dashboard
await risk_dashboard(account_id)

## Summary and Best Practices

Congratulations! You've learned comprehensive risk management techniques:

### ✅ **Key Risk Management Concepts Covered**

1. **Position Sizing**: Calculate optimal position sizes based on risk tolerance
2. **Stop Losses**: Implement automatic stop losses to limit downside
3. **Take Profits**: Set profit targets for systematic gains
4. **Portfolio Risk**: Monitor overall portfolio exposure
5. **Daily Limits**: Prevent catastrophic daily losses
6. **Advanced Orders**: Bracket orders, trailing stops, scaling out
7. **Risk Scoring**: Quantitative risk assessment

### 🛡️ **Risk Management Best Practices**

- **Never risk more than 2% per trade**
- **Use stop losses on every trade**
- **Maintain risk-reward ratios of at least 1.5:1**
- **Diversify across different currency pairs**
- **Monitor portfolio correlation**
- **Set daily and weekly loss limits**
- **Review and adjust risk parameters regularly**

### ⚠️ **Critical Reminders**

- 🚨 **Risk management is more important than profit generation**
- 📊 **Always validate trades before execution**
- 🔄 **Monitor positions continuously**
- 📈 **Adjust position sizes based on market volatility**
- 🛑 **Never trade without stop losses**

### 🎯 **Next Steps**

- Explore [Data Analysis](data-analysis.ipynb) for backtesting risk models
- Check out [Streaming Data](streaming-data.ipynb) for real-time risk monitoring
- Review [Trading Strategies](trading-strategies.ipynb) with risk-aware implementations
- Study the [User Guide](../../user-guide/best-practices.md) for production risk management